# DML (ROBUST)
This script runs double machine learning algorythms (partially-linear models). With FEs and clustered SEs.

**Motivation**

The DML estimates so far assume independent observations. In a dyadic trade panel
this is violated on two fronts: each country appears in many pairs (shocks to one
country are shared across all its dyads), and each pair is observed over many years
(serial correlation within a dyad). Treating the ~1.4M observations as independent
therefore distorts the standard errors.

We re-estimate the fixed-effects DML specifications with standard errors **clustered
by country-pair**, which allows arbitrary correlation of the errors within each dyad
over time. The point estimates are unchanged — clustering affects only inference. This provides an honest
significance test for the aggregate sanction effect, accounting for the dependence
structure of the data.

#### Libraries

In [ ]:
!pip install -q doubleml pyfixest
import numpy as np, pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pyfixest as pf
import doubleml as dml
from doubleml import DoubleMLData, DoubleMLPLR

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.8/607.8 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.7/531.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.2/

In [ ]:
SEED = 123

Connect to Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'
export_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'

Data uploading

In [ ]:
# ── Load panel + define columns (same convention) ──
import_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'
merged_robust = pd.read_csv(import_path + "merged_robust_final.csv",
                            dtype={"exp_iso3": str, "imp_iso3": str})
merged_main = merged_robust   # alias — downstream cells unchanged

sanc_feats = [
    "sanc_arms","sanc_military","sanc_trade","sanc_financial","sanc_travel","sanc_other",
    "target_mult","sender_mult",
    "descr_exp_compl","descr_exp_part","descr_imp_compl","descr_imp_part",
    "obj_democracy","obj_destab_regime","obj_end_war","obj_human_rights","obj_other",
    "obj_policy_change","obj_prevent_war","obj_territorial_conflict","obj_terrorism",
]
grav_feats = ["dist_w_harm","contig","comlang","comcol","colony",
              "exp_gdp","imp_gdp","exp_pop","imp_pop",
              "fta","exp_eu","imp_eu","exp_wto","imp_wto"]
print(merged_main.shape)

/tmp/ipykernel_1311/277102000.py:3: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_robust = pd.read_csv(import_path + "merged_robust_final.csv",


(1404170, 54)


## DML modeling

In [ ]:
# ── FE residualization (produces y_res, D_res) ──
fe = merged_main[["exp_iso3","imp_iso3","year"]].copy()
fe["y"]    = np.log1p(merged_main["trade"].values)
fe["D"]    = merged_main["sanctioned_any"].values.astype(float)
fe["ey"]   = fe["exp_iso3"] + "_" + fe["year"].astype(str)
fe["iy"]   = fe["imp_iso3"] + "_" + fe["year"].astype(str)
fe["pair"] = fe["exp_iso3"] + "_" + fe["imp_iso3"]

y_res = np.asarray(pf.feols("y ~ 1 | ey + iy + pair", data=fe).resid())
D_res = np.asarray(pf.feols("D ~ 1 | ey + iy + pair", data=fe).resid())
print("D_res var:", round(float(D_res.var()), 6))

/usr/local/lib/python3.13/dist-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 6 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 6 singleton fixed effect(s) dropped from the model.
  warnings.warn(


D_res var: 0.008505


In [ ]:
# ── align W to kept rows (pyfixest drops 6 singletons) + impute ──
from sklearn.impute import SimpleImputer
keep_mask = ((fe.groupby("ey")["ey"].transform("size") > 1) &
             (fe.groupby("iy")["iy"].transform("size") > 1) &
             (fe.groupby("pair")["pair"].transform("size") > 1))
mm = merged_main[keep_mask.values].reset_index(drop=True)

econ  = ["exp_gdp","imp_gdp","exp_pop","imp_pop","dist_w_harm"]
flags = ["contig","comlang","comcol","colony","fta","exp_eu","imp_eu","exp_wto","imp_wto"]
Wdf = mm[econ + flags].copy()
for c in econ: Wdf[c] = np.log1p(Wdf[c])
Wdf[econ]  = SimpleImputer(strategy="median").fit_transform(Wdf[econ])
Wdf[flags] = Wdf[flags].fillna(0)
W_fe = Wdf.values
assert not np.isnan(W_fe).any()
print("aligned:", len(y_res), len(D_res), len(W_fe))

aligned: 1404164 1404164 1404164


### Clustering data

In [ ]:
# ── clustered DoubleML data ──
dml_df = pd.DataFrame(W_fe, columns=econ + flags)
dml_df["y_res"] = y_res
dml_df["D_res"] = D_res
dml_df["pair_id"] = (mm["exp_iso3"] + "_" + mm["imp_iso3"]).astype("category").cat.codes

dml_data = DoubleMLData(
    dml_df, y_col="y_res", d_cols="D_res",
    x_cols=econ + flags,
    cluster_cols="pair_id"          # pair-clustered inference
)
print(dml_data)

================== DoubleMLData Object ==================

------------------ Data summary      ------------------
Outcome variable: y_res
Treatment variable(s): ['D_res']
Covariates: ['exp_gdp', 'imp_gdp', 'exp_pop', 'imp_pop', 'dist_w_harm', 'contig', 'comlang', 'comcol', 'colony', 'fta', 'exp_eu', 'imp_eu', 'exp_wto', 'imp_wto']
Instrument variable(s): None
Cluster variable(s): ['pair_id']
Is cluster data: True
No. Observations: 1404164
------------------ DataFrame info    ------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1404164 entries, 0 to 1404163
Columns: 17 entries, exp_gdp to pair_id
dtypes: float64(16), int32(1)
memory usage: 176.8 MB



## FE Lasso, SE-clustered

In [ ]:
%%time
# ── FE-Lasso, pair-clustered  ──
np.random.seed(SEED)
plr_lasso = DoubleMLPLR(
    dml_data,
    ml_l = make_pipeline(StandardScaler(), LassoCV(random_state=SEED)),
    ml_m = make_pipeline(StandardScaler(), LassoCV(random_state=SEED)),
    n_folds = 3
)
plr_lasso.fit()
print(plr_lasso.summary)

           coef   std err         t     P>|t|     2.5 %    97.5 %
D_res -0.026136  0.025184 -1.037823  0.299352 -0.075495  0.023223
CPU times: user 1min 26s, sys: 4.78 s, total: 1min 31s
Wall time: 1min 22s


Under pair-clustered inference, the fixed-effects Lasso DML estimate of the aggregate sanction effect is negative but statistically insignificant (−0.024, p = 0.30). The near-doubling of the standard error relative to independence-based inference shows that the earlier marginal significance was an artifact of ignoring within-dyad error correlation.

## FE Random Forest, SE-clustered

In [ ]:
np.random.seed(SEED)
plr_rf = DoubleMLPLR(
    dml_data,
    ml_l = RandomForestRegressor(n_estimators=100, min_samples_leaf=50,
                                 max_samples=0.5, random_state=SEED, n_jobs=-1),
    ml_m = RandomForestRegressor(n_estimators=100, min_samples_leaf=50,
                                 max_samples=0.5, random_state=SEED, n_jobs=-1),
    n_folds = 3
)
plr_rf.fit()                                          # the ~50-min run

import joblib
joblib.dump(plr_rf, import_path + "plr_rf_clustered_rob.pkl")   # save after fitting
print(plr_rf.summary)

           coef   std err         t     P>|t|     2.5 %    97.5 %
D_res -0.067399  0.027839 -2.421074  0.015475 -0.121962 -0.012837


In [ ]:
## to recover the model results if needed
# plr_rf = joblib.load(import_path + "plr_rf_clustered_rob.pkl")

With fixed effects partialled out and standard errors clustered by country-pair, the two nuisance specifications diverge sharply. The Lasso estimate is negative but insignificant (−0.024, p = 0.30): once within-dyad error correlation is accounted for, its standard error nearly doubles and the effect cannot be distinguished from zero. The Random Forest estimate, by contrast, remains negative and significant (−0.053, p = 0.037, 95% CI [−0.102, −0.003]) despite the same increase in standard error. The aggregate sanction effect is thus recovered only when flexible, non-linear nuisance models are combined with fixed effects and honest clustered inference — the linear learner misses the non-linear selection into sanctions that the forest captures.

## Output tables

In [ ]:
# helper functions
from scipy.stats import norm

def plr_row(model, nuisance):
    s = model.summary.iloc[0]
    b, se = float(s["coef"]), float(s["std err"])
    return {"Nuisance": nuisance, "b": b, "se": se,
            "t": b/se, "p": 2*(1-norm.cdf(abs(b/se))), "pct": 100*(np.exp(b)-1)}

dml_tab = pd.DataFrame([
    plr_row(plr_lasso, "Lasso"),
    plr_row(plr_rf,    "Random Forest"),
])
dml_tab["stars"] = pd.cut(dml_tab.p, [-np.inf,.01,.05,.1,np.inf],
                          labels=["***","**","*",""]).astype(str)
dml_tab

,Nuisance,b,se,t,p,pct,stars
0,Lasso,-0.026136,0.025184,-1.037823,0.299352,-2.579767,
1,Random Forest,-0.067399,0.027839,-2.421074,0.015475,-6.517833,**


In [ ]:
# collab view
def dml_show(df):
    d = df.copy()
    d["β"]        = d.apply(lambda r: f"{r.b:.4f}{r.stars}", axis=1)
    d["SE"]       = d.se.map(lambda x: f"({x:.3f})")
    d["% effect"] = d.pct.map(lambda x: f"{x:+.1f}")
    d["t"]        = d.t.map(lambda x: f"{x:.2f}")
    return (d[["Nuisance","β","SE","% effect","t"]]
            .style.hide(axis="index")
            .set_caption("Table 4: FE-DML, pair-clustered SEs")
            .set_table_styles([{"selector":"caption",
                "props":[("font-weight","bold"),("font-size","13px"),("padding","6px")]}]))

dml_show(dml_tab)

Nuisance,β,SE,% effect,t
Lasso,-0.0261,(0.025),-2.6,-1.04
Random Forest,-0.0674**,(0.028),-6.5,-2.42


In [ ]:
# latex
def dml_latex(df):
    L = [r"\begin{table}[ht]", r"\centering",
         r"\caption{Fixed-effects DML with pair-clustered standard errors}",
         r"\label{tab:dml_clustered}",
         r"\begin{tabular}{lccc}", r"\toprule",
         r"Nuisance model & $\beta$ & \% effect & $t$ \\", r"\midrule"]
    for _, r in df.iterrows():
        L.append(f"{r.Nuisance} & {r.b:.4f}{r.stars} & {r.pct:+.1f} & {r.t:.2f} \\\\")
        L.append(f" & ({r.se:.3f}) &  &  \\\\")
    L += [r"\midrule",
          r"\multicolumn{4}{l}{\footnotesize Outcome $\log(1+\text{trade})$; treatment = any sanction.}\\",
          r"\multicolumn{4}{l}{\footnotesize Exporter-year, importer-year, pair FE partialled out;}\\",
          r"\multicolumn{4}{l}{\footnotesize SE clustered by pair, in parentheses. *** p$<$.01, ** p$<$.05, * p$<$.1}\\",
          r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    print("\n".join(L))

dml_latex(dml_tab)

\begin{table}[ht]
\centering
\caption{Fixed-effects DML with pair-clustered standard errors}
\label{tab:dml_clustered}
\begin{tabular}{lccc}
\toprule
Nuisance model & $\beta$ & \% effect & $t$ \\
\midrule
Lasso & -0.0261 & -2.6 & -1.04 \\
 & (0.025) &  &  \\
Random Forest & -0.0674** & -6.5 & -2.42 \\
 & (0.028) &  &  \\
\midrule
\multicolumn{4}{l}{\footnotesize Outcome $\log(1+\text{trade})$; treatment = any sanction.}\\
\multicolumn{4}{l}{\footnotesize Exporter-year, importer-year, pair FE partialled out;}\\
\multicolumn{4}{l}{\footnotesize SE clustered by pair, in parentheses. *** p$<$.01, ** p$<$.05, * p$<$.1}\\
\bottomrule
\end{tabular}
\end{table}
